# 用 Reflect 檢查回答是否有依據

## 在 Colab 準備環境

如果你是在 Colab 開啟，先複製專案並切換到專案資料夾；如果你已經在專案資料夾，可以直接跳過這格。

In [ ]:
!git clone https://github.com/R300-AI/Agentic-SDK.git
%cd Agentic-SDK

## 載入流程元件

這裡先用 `KeywordRetrieve` 建立一個最小、可重現的查詢來源。接下來會比較：只有 action 時會得到什麼；加上 Reflect 後，結果多了哪些可檢查訊號。

In [ ]:
from agentic_sdk import Workflow
from agentic_sdk.modules import DirectAnswerAction, EvidenceCheckReflect, KeywordRetrieve, PassThroughPerceive

## 用最小資料建立 evidence source

為了讓 Reflect 的差異容易觀察，這裡先把資料來源縮到兩筆固定條目。這樣後面看到的 pass/fail 會來自流程狀態，而不是搜尋模型的不確定性。

In [ ]:
reference_items = [
    {'keywords': ['保存', 'bundle', '參考文件'], 'content': '使用參考文件的 Agent 需要保存 bundle，重新打開時才找得到原本的資料。'},
    {'keywords': ['runner', '唯讀'], 'content': '公開分享的 Runner 可以使用 Agent，但不能修改設定或保存。'},
]

retrieve = KeywordRetrieve(items=reference_items, fallback='目前沒有找到相關參考資料。')

## 加上 Reflect 驗收關卡

先建立兩個 workflow：一個沒有 Reflect，另一個加上 `EvidenceCheckReflect(on_failure='end')`。接下來用同一個查不到資料的問題跑兩次，觀察加上 Reflect 後結果多了哪些欄位與紀錄。

In [ ]:
workflow_without_reflect = Workflow(
    workflow_name='No reflect baseline Agent',
    perceive=PassThroughPerceive(),
    retrieve=retrieve,
    action=DirectAnswerAction(),
)

workflow = Workflow(
    workflow_name='Reflect evidence check Agent',
    perceive=PassThroughPerceive(),
    retrieve=retrieve,
    action=DirectAnswerAction(),
    reflect=EvidenceCheckReflect(on_failure='end'),
)

## Case 1：問題命中參考資料

這題會命中 bundle 條目。執行後先看回答，再看 `reflect_verdict` 是否顯示這次驗收通過。

In [ ]:
grounded_result = workflow.run('為什麼使用參考文件的 Agent 要保存 bundle？')
print(grounded_result.final_message)
print('reflect verdict:', grounded_result.entities.get('reflect_verdict'))

## 看 Reflect 留下的驗收紀錄

`reflect_verdict` 適合給程式判斷；reflection entry 則適合除錯，因為它會記錄 verdict 和 reason。

In [ ]:
reflection_entries = [entry for entry in grounded_result.entries if entry.type == 'reflection']
print(reflection_entries[-1].content)
print(reflection_entries[-1].metadata)

## Case 2：同一題，比較沒有 Reflect 與有 Reflect

沒有 Reflect 時，workflow 只會回傳 fallback 訊息。加上 `EvidenceCheckReflect` 後，差異不在 fallback 文字，而是在結果裡多了一個機器可讀的 `fail` 訊號。應用程式可以用這個訊號決定是否顯示、重試或要求更多資料。

In [ ]:
missing_question = '這個 Agent 支援哪些 GPU driver 版本？'

missing_without_reflect = workflow_without_reflect.run(missing_question)
missing_result = workflow.run(missing_question)

print('[without Reflect]', missing_without_reflect.final_message)
print('reflect verdict:', missing_without_reflect.entities.get('reflect_verdict'))
print()
print('[with EvidenceCheckReflect]', missing_result.final_message)
print('reflect verdict:', missing_result.entities.get('reflect_verdict'))
print('visit counts:', missing_result.visit_counts)

## 看沒有 evidence 時的紀錄

從這兩筆紀錄可以看到差異：retrieve 先記下 `hit_count: 0`，Reflect 再把這次回答標成 `fail`。

In [ ]:
for entry in missing_result.entries:
    if entry.type in {'retrieved', 'reflection'}:
        print(entry.type, entry.content, entry.metadata)

## `on_failure` 的實際用途：決定失敗後去哪裡

`on_failure='end'` 代表驗收失敗就停在目前結果；`on_failure='retry_plan'` 代表驗收失敗後回到 plan，讓流程有機會調整查詢、重取 evidence、再回答一次。為了把 routing 看清楚，下面不用 LLM，而是寫一個固定規則的 plan：第一次用原問題查詢；如果 Reflect 失敗，第二次改查一個資料裡存在的 bundle 問題。

In [ ]:
class RetryWithBundlePlan:
    name = 'plan'

    def __call__(self, state):
        failed_before = any(
            entry.type == 'reflection' and entry.metadata.get('verdict') == 'fail'
            for entry in state.entries
        )
        if failed_before:
            return {
                'next_module': 'retrieve',
                'payload': {
                    'query': '為什麼使用參考文件的 Agent 要保存 bundle？',
                    'plan_thought': 'previous evidence check failed; retry with a documented bundle question',
                },
            }
        return {
            'next_module': 'retrieve',
            'payload': {
                'query': state.latest_user_message(),
                'plan_thought': 'try the original user question first',
            },
        }

retry_workflow = Workflow(
    workflow_name='Reflect retry plan Agent',
    perceive=PassThroughPerceive(),
    plan=RetryWithBundlePlan(),
    retrieve=retrieve,
    action=DirectAnswerAction(),
    reflect=EvidenceCheckReflect(on_failure='retry_plan'),
)

retry_result = retry_workflow.run(missing_question)
print(retry_result.final_message)
print('reflect verdict:', retry_result.entities.get('reflect_verdict'))
print('visit counts:', retry_result.visit_counts)

for entry in retry_result.entries:
    if entry.type == 'reflection':
        print(entry.content, entry.metadata)

## 這一章的重點

讀完這章後，應該能分辨三件事：沒有 Reflect 時只得到 action 結果；加上 `EvidenceCheckReflect` 後會多出 `reflect_verdict` 和 reflection entry；`on_failure` 決定驗收失敗後是停下來，還是回到 plan 重試。